In [ ]:
import time
import numpy as np
import torch
from transformers import RobertaForSequenceClassification, RobertaTokenizer

In [ ]:
short_text  = "Oh brilliant, just what I needed."
medium_text = "The service was absolutely fantastic, I would recommend to everyone."
long_text   = medium_text * 5

In [ ]:
def time_model(predict_fn, text, n_runs=20):
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        predict_fn(text)
        end = time.perf_counter()
        times.append((end - start) * 1000)
    return {
        "mean_ms": round(np.mean(times), 2),
        "std_ms":  round(np.std(times), 2)
    }

In [ ]:
model = RobertaForSequenceClassification.from_pretrained(
    "joela0/besstie-roberta-all-pool"
)
tokenizer = RobertaTokenizer.from_pretrained(
    "joela0/besstie-roberta-all-pool"
)
model.eval()

In [ ]:
def predict_roberta(text):
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=128
    )
    with torch.no_grad():
        outputs = model(**inputs)
    return torch.argmax(outputs.logits).item()

print("Short:", time_model(predict_roberta, short_text))
print("Medium:", time_model(predict_roberta, medium_text))
print("Long:", time_model(predict_roberta, long_text))